Część cyklu o wykorzystaniu rzeczywistych danych i budowaniu z nich modelu.

In [1]:
# standardowe przetwarzanie danych
import pandas as pd

import joblib

# przetwarzanie danych geograficznych
import geopandas as gpd
from shapely.geometry import Point
from geopy.distance import geodesic

# wczytujemy model
model = joblib.load("model_all.pkl")

# Użycie modelu

In [4]:

data = {
    "area": 55.5,
    "balcony": 8,
    "balcony_and_loggia": 8,
    "distance_to_city_center": 1.4,
    "floor": 4,
    "floor_max": 6,
    "floors": 1,
    "garden": 0,
    "gmina_area": 120,
    # to jest wyliczone gmina_area / ludnosc:
    "gmina_area_per_person": 0.0006,
    "gmina_rodzaj": 1,
    "has_loggia": 0,
    "has_terrace": 0,
    "lat": 52.42,
    "lng": 16.92,
    "ludnosc": 200_000,
    "terrace_and_garden": 0,
    "top_floor": 0,
    "wynagrodzenie": 7500,
}


XGBoost musi mieć kolumny w odpowiedniej kolejności.

In [3]:
# Pobranie listy cech, na których model był trenowany
# oraz ich kolejności
feature_names = model.get_booster().feature_names
feature_names

['area',
 'lng',
 'lat',
 'floor',
 'floors',
 'balcony',
 'garden',
 'floor_max',
 'balcony_and_loggia',
 'terrace_and_garden',
 'gmina_area',
 'distance_to_city_center',
 'gmina_rodzaj',
 'wynagrodzenie',
 'ludnosc',
 'gmina_area_per_person']

In [ ]:
#df z naszego słownika
predict_df = pd.DataFrame([data])

#ustawiamy kolumny wg kolejności
predict_df = predict_df[feature_names]


In [ ]:
#predykcja ceny metra kwardratowego z modelu:
pred = model.predict(predict_df)
pred

array([12065.25], dtype=float32)

## Uwzględnijmy słowniki i nazwę gminy

In [8]:
slownik_gmin_centrum_gdf = gpd.read_file("slownik_gmin_centrum.gpkg")
slownik_gmin_obrys_gdf = gpd.read_file("slownik_gmin_obrys.gpkg")

In [9]:
print(slownik_gmin_centrum_gdf)

                gmina_nazwa woj_teryt gmina_teryt                   geometry
0                 Białaczów        10     1007013   POINT (20.30089 51.2966)
1                    Bircza        18     1813013  POINT (22.43453 49.68833)
2                Bobrowniki        04     0408023  POINT (19.00078 52.78111)
3                   Bogoria        26     2612013  POINT (21.27359 50.65496)
4               Bolesławiec        10     1018013  POINT (18.21407 51.21476)
...                     ...       ...         ...                        ...
2491   Targówek - dzielnica        14     1465118  POINT (21.04782 52.22929)
2492    Ursynów - dzielnica        14     1465138  POINT (21.04782 52.22929)
2493      Wawer - dzielnica        14     1465148  POINT (21.04782 52.22929)
2494     Wesoła - dzielnica        14     1465158  POINT (21.04782 52.22929)
2495  Rembertów - dzielnica        14     1465098  POINT (21.04782 52.22929)

[2496 rows x 4 columns]


## Funkcje do szukania właściwej gminy

In [25]:
def find_gmina_teryt(map_df,lat, lng):
    """
    Znajdź gminę w której znajduje się dany punkt
    """

    point = Point(lng, lat)

    for idx, row in map_df.iterrows():
        if row["geometry"].contains(point):
            print("Nazwa gminy: ",row["gmina_nazwa"])
            return row["gmina_teryt"]
    #jeśli gminy nie ma w żadnym obszarze 
    return None

def get_distance_to_city_center(map_df, teryt, lat, lng):
    """
    Oblicza odległość do centrum miasta
    """

    center = map_df[map_df["gmina_teryt"]==teryt]["geometry"].values[0]
    return geodesic((center.y, center.x),(lat,lng)).km

def get_gus_data(map_df, teryt):
    """
    Odczytanie danych z konktetnej gminy
    """
    teryt_data = map_df[map_df["gmina_teryt"]==teryt]
    if teryt_data.shape[0]:
        return{
            "ludnosc": teryt_data["ludnosc"].values[0],
            "wynagrodzenie": teryt_data["wynagrodzenie"].values[0],
            "gmina_area": teryt_data["gmina_area"].values[0],
            "gmina_area_per_person": teryt_data["gmina_area"].values[0] / teryt_data["ludnosc"].values[0],
            "gmina_rodzaj": teryt_data["gmina_rodzaj"].values[0],        
        }
    
    return {
            "ludnosc": None,
            "wynagrodzenie": None,
            "gmina_area": None,
            "gmina_area_per_person": None,
            "gmina_rodzaj": None,
        }

# Obliczenia
## Dane wejściowe

In [11]:
data = {
    "lng": 20.9524065,
    "lat": 52.3391106,
    "area": 72.7,
    "balcony": 0,
    "floor": 8,
    "floor_max": 8,
    "floors": 1,
    "garden": 0,
    "loggia": 11,
    "terrace": 17,
}

## Dane wyliczane

In [12]:
data["top_floor"] = 1 if data["floor"] == data["floor_max"] else 0
data["has_loggia"] = 1 if data["loggia"] > 0 else 0
data["has_terrace"] = 1 if data["terrace"] > 0 else 0
data["balcony_and_loggia"] = data["balcony"] + data["loggia"]
data["terrace_and_garden"] = data["terrace"] + data["garden"]
# zdejmujemy ze słownika zbędne dla modelu cechy
data.pop("loggia")
data.pop("terrace")

# kod teryt gminy, w której leży wskazany punkt
gm_teryt = find_gmina_teryt(slownik_gmin_obrys_gdf, data["lat"], data["lng"])

# dane ze słowników
data["distance_to_city_center"] = get_distance_to_city_center(
    slownik_gmin_centrum_gdf, gm_teryt, data["lat"], data["lng"]
)
data_gus = get_gus_data(slownik_gmin_obrys_gdf, gm_teryt)

# uzupełnienie całego zapytania do modelu = suma słowników
data = data | data_gus

## Użycie powyższych danych do modelu

In [13]:
feature_names = model.get_booster().feature_names

predict_df = pd.DataFrame([data])
predict_df = predict_df[feature_names]

In [19]:
price_m2 = model.predict(predict_df)[0]
print("Cena za m2:", price_m2)

price = price_m2 * data["area"]
print("Cena całkowita", price)

Cena za m2: 15138.252
Cena całkowita 1100550.9


## Stworzenie funkcji do obliczania

In [23]:
def calculate_price(data):
    """
    Oblicz cenę nieruchomości bazując na danych wejściowych
    """
    data["top_floor"] = 1 if data["floor"] == data["floor_max"] else 0
    data["has_loggia"] = 1 if data["loggia"] > 0 else 0
    data["has_terrace"] = 1 if data["terrace"] > 0 else 0
    data["balcony_and_loggia"] = data["balcony"] + data["loggia"]
    data["terrace_and_garden"] = data["terrace"] + data["garden"]
    # zdejmujemy ze słownika zbędne dla modelu cechy
    data.pop("loggia")
    data.pop("terrace")

    # kod teryt gminy, w której leży wskazany punkt
    gm_teryt = find_gmina_teryt(slownik_gmin_obrys_gdf, data["lat"], data["lng"])

    # dane ze słowników
    data["distance_to_city_center"] = get_distance_to_city_center(
        slownik_gmin_centrum_gdf, gm_teryt, data["lat"], data["lng"]
    )
    data_gus = get_gus_data(slownik_gmin_obrys_gdf, gm_teryt)

    # uzupełnienie całego zapytania do modelu = suma słowników
    data = data | data_gus
    
    feature_names = model.get_booster().feature_names

    predict_df = pd.DataFrame([data])
    predict_df = predict_df[feature_names]

    price_m2 = model.predict(predict_df)[0]
    print("Cena za m2:", price_m2)

    price = price_m2 * data["area"]
    print("Cena całkowita", price)

    return price_m2, price

In [29]:
data = {
    "lng": 16.94811638324749,
    "lat": 52.43798478167113, 
    "area": 122.22,
    "balcony": 5.3,
    "floor": 3,
    "floor_max": 4,
    "floors": 1,
    "garden": 0,
    "loggia": 0,
    "terrace": 0,
}

_,_ = calculate_price(data)

Nazwa gminy:  Poznań
Cena za m2: 13621.374
Cena całkowita 1664804.4


# Pomysły na przyszłość

W ramach ćwiczeń proponuję:

- postaraj się wypełnić braki w oryginalnych danych, zamiast usuwać całe rekordy (robiliśmy to w odcinku pierwszym)
-dodaj kilka dodatkowych wymiarów, na przykład korzystając z danych GUS - mogą to być dane związane z demografią typu średni wiek (użyłbym bardziej struktury wiekowej i procentowych udziałów poszczególnych grup wiekowych) mieszkańców gminy, albo makroekonomiczne typu stopa bezrobocia, zatrudnienie w poszczególnych sektorach gospodarki (bo może one charakteryzują się innym poziomem wynagrodzenia?)
- dodaj agregaty (trzeba zbudować słowniki) - ile kosztuje metr mieszkania w gminie? a konkretnie na 5 piętrze w tej gminie? ile kosztuje mieszkanie od tego dewelopera? w tym powiecie i ogólnie w kraju? Jeśli dodasz agregaty to pewnie będzie trzeba je zesłownikować i doklejać przed predykcją - użytkownik nie powinien ich podawać, bo skąd ma je znać?
- korzystając z dostępnych publicznie danych (np. OpenStreetMap) można poszukać najbliższej szkoły, sklepu, przystanku autobusowego (czy stacji metra w Warszawie) do nieruchomości - to powinno mieć wpływ na cenę. Aby skorzystać z danych OpenStreetMap przyda się pakiet OSMnx
- rozbij model na wiele mniejszych, każdy wyucz oddzielnie (może nawet na innych cechach - odległość od metra nie ma sensu poza Warszawą)
- być może pojawi się zbiór danych z innego czasu (np. rok później) - wówczas znaczenie będzie miała dynamika zmian cen; dobrą robotę robi tutaj OlxData